[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/limorigu/uchicago-python-ai-workshop/blob/main/notebooks/01_python_bootcamp.ipynb)

# 01 · Python bootcamp — Labs A, B, C (Day 1)

One notebook, three labs, matched to the three morning/early-afternoon talks:

| Lab | When | After the talk on | Running example |
|---|---|---|---|
| **A** | 9:55–10:30 | values, variables, types | a 1-day VaR calculation |
| **B** | 11:20–12:00 | lists, dicts, functions, loops | a five-position book |
| **C** | 1:30–2:10 | the DataFrame, Excel→pandas | the FCC broadband data |

**How to work:** run each cell, read the output, then do the `# TODO` in the cell below it. Cells marked **⚡ Fast lane** are optional extras for people who have coded before — skip them if you are new.

**Ask for help early.** Raise a hand; an error message is a normal part of programming, not a sign you did it wrong. Read error messages from the **bottom line up**.

In [1]:
# --- Setup: get the workshop files into this Colab session ------------------------------
# Downloads the workshop repository (data + helper code). Takes ~10 seconds the first time.
import os, sys
REPO_URL = "https://github.com/limorigu/uchicago-python-ai-workshop"
if not any(os.path.exists(d) for d in ["workshop/data", "data", "../data"]):
    !GIT_TERMINAL_PROMPT=0 git clone -q --depth 1 $REPO_URL workshop
DATA = next((d for d in ["workshop/data", "data", "../data"] if os.path.exists(d)), None)
if DATA is None:
    raise RuntimeError("Could not download the workshop files. Check your internet connection and "
                       "that this link opens in your browser: " + REPO_URL + " . Still stuck? Email the instructor.")
sys.path.insert(0, "workshop/app" if os.path.exists("workshop/app") else "../app")
print("Data folder:", DATA, "| files:", sorted(os.listdir(DATA)))

Data folder: workshop/data | files: ['case1', 'case2']


---
# Lab A · Values, variables, types (9:55–10:30)

Excel cells hold values. Python **variables** hold values *and have names you choose*. The name is the documentation.

In the talk you saw a parametric **Value-at-Risk** built from four variables. Here you rebuild it yourself, then do what the PM always does: change the assumptions.

In [56]:
# A parametric 1-day VaR, from nothing but variables.
position_value = 25_000_000      # a number (int) — $25M position. Underscores are thousands separators.
daily_vol = 0.018                # a decimal (float) — 1.8% daily volatility
z_99 = 2.33                      # 99% confidence z-score
desk = "Consumer & Lodging"      # text (str)
is_liquid = True                 # a boolean

var_1d = position_value * daily_vol * z_99
print(f"{desk}: 1-day 99% VaR = ${var_1d:,.0f}")
print(type(position_value), type(daily_vol), type(desk), type(is_liquid))

Consumer & Lodging: 1-day 99% VaR = $1,048,500
<class 'int'> <class 'float'> <class 'str'> <class 'bool'>


In [58]:
# TODO 1: the PM asks for 95% confidence instead of 99% (z = 1.65).
#         Add a variable z_95 and compute var_1d_95. Which assumption did you change? How many places?
z_95 = 1.65
var_1d_95 = position_value * daily_vol * z_95
print(f"{desk}: 1-day 95% VaR = ${var_1d_95:,.0f}")
# TODO 2: the position grows to $40M. Change ONE line above and re-run both cells.
#         Notice: like Excel, everything downstream updates — but the assumption has a NAME and ONE home.
position_value = 40_000_000
var_1d_95 = position_value * daily_vol * z_95
print(f"{desk}: 1-day 99% VaR = ${var_1d:,.0f}")
var_1d = position_value * daily_vol * z_99
print(f"{desk}: 1-day 95% VaR = ${var_1d_95:,.0f}")
print(type(position_value), type(daily_vol), type(desk), type(is_liquid))



Consumer & Lodging: 1-day 95% VaR = $742,500
Consumer & Lodging: 1-day 99% VaR = $1,048,500
Consumer & Lodging: 1-day 95% VaR = $1,188,000
<class 'int'> <class 'float'> <class 'str'> <class 'bool'>


### Strings and numbers do not mix silently
Excel will happily treat `"5"` and `5` as the same thing until, one day, it doesn't (every analyst has met a ZIP code or CUSIP turned into a number). Python refuses up front:

In [48]:
print("5" + "5")        # text + text = text: '55'
print(5 + 5)            # number + number: 10
try:
    print("5" + 5)      # text + number: an error — on purpose
except TypeError as e:
    print("TypeError:", e)

ticker = "HLT"
shares = 120_000
price = 61.42
print(f"{ticker}: {shares:,} shares × ${price} = ${shares * price:,.0f}")   # f-strings format like a tear sheet

55
10
TypeError: can only concatenate str (not "int") to str
HLT: 120,000 shares × $61.42 = $7,370,400


In [49]:
# TODO: compute the simple daily return of a stock that closed at 61.42 yesterday and 62.05 today,
#       and print it as a percentage with 2 decimals (hint: f"{r:.2%}"). Expected: 1.03%
price_yesterday = 61.42
price_today = 62.05
r = (price_today - price_yesterday) / price_yesterday
print(f"{r:.2%}" )


1.03%


## ⚡ Lab A fast lane (optional)
1. Scale the 1-day VaR to a 10-day horizon using the square-root-of-time rule: `var_10d = var_1d * 10 ** 0.5`.
2. Write the same with `math.sqrt` (`import math`). Which reads better?
3. At what position size does 1-day 99% VaR exceed a $1.5M risk limit? Solve it by algebra, then verify by changing the variable.

In [57]:
var_10d = var_1d * 10 ** 0.5
print(f"{desk}: 10-day 99% VaR = ${var_10d:,.0f}")

import math
var_10d = var_1d * math.sqrt(10)
print(f"{desk}: 10-day 99% VaR = ${var_10d:,.0f}")

limit = 1_500_000
max_position = limit / (daily_vol * z_99)
print(f"Max position : ${max_position:,.0f}")

Consumer & Lodging: 10-day 99% VaR = $3,315,648
Consumer & Lodging: 10-day 99% VaR = $3,315,648
Max position : $35,765,379


## Lab A check-out (2 min)
In the cell below, one sentence: *where do assumptions live in your Excel models today — and who else can find them?*

In Excel my assumptions just sit in regular cells, and someone can just find it by clicking the cells

---
# Lab B · Lists, dictionaries, functions, loops (11:20–12:00)

A **list** is an ordered column of values. A **dictionary** is a lookup table (key → value) — the thing you reach for VLOOKUP for. A **function** is a named formula with documentation. A **loop** is fill-down.

Running example: a five-position book.

In [6]:
positions = ["HLT", "MAR", "H", "CHH", "WH"]            # a list: an ordered column
print(positions[0], "|", positions[-1], "|", positions[1:3], "|", len(positions))

book = {                                                  # a dictionary: ticker -> position value ($)
    "HLT": 25_000_000, "MAR": 18_000_000, "H": 9_500_000, "CHH": 4_200_000, "WH": 6_800_000,
}
print("HLT position:", f"${book['HLT']:,}")
book["HLT"] = 27_500_000                                  # update a value
print("Book size:", f"${sum(book.values()):,}")

HLT | WH | ['MAR', 'H'] | 5
HLT position: $25,000,000
Book size: $66,000,000


In [7]:
# TODO 1: add a new position "IHG" of $5_000_000 to the dictionary and reprint the book size.
# TODO 2: positions.append(...) the ticker "IHG" to the list too, and print the new length.

### Functions: define once, reuse, test

In [8]:
def position_pnl(value_start, value_end):
    """Dollar P&L and simple return for one position."""
    pnl = value_end - value_start
    ret = pnl / value_start
    return pnl, ret

pnl, ret = position_pnl(25_000_000, 25_800_000)
print(f"P&L ${pnl:,.0f} ({ret:.2%})")

P&L $800,000 (3.20%)


In [9]:
# TODO: write a function position_var(value, daily_vol, z=2.33) that returns value * daily_vol * z.
#       Call it on the HLT position with daily_vol=0.018. Expected: about $1,153,575.

### Loops and conditions: fill-down and IF

In [10]:
vols = {"HLT": 0.018, "MAR": 0.016, "H": 0.022, "CHH": 0.019, "WH": 0.025}
POSITION_LIMIT = 20_000_000

for ticker, value in book.items():
    flag = "⚠ LIMIT BREACH" if value > POSITION_LIMIT else ""
    print(f"{ticker:4s} ${value:>12,}  vol {vols.get(ticker, 0):.1%}  {flag}")

HLT  $  27,500,000  vol 1.8%  ⚠ LIMIT BREACH
MAR  $  18,000,000  vol 1.6%  
H    $   9,500,000  vol 2.2%  
CHH  $   4,200,000  vol 1.9%  
WH   $   6,800,000  vol 2.5%  


In [11]:
# TODO 1: loop over the book and print each position's 1-day 99% VaR, using your position_var function.
# TODO 2: sum the position VaRs into total_var as you loop (start with total_var = 0).
#         (Yes, summing VaRs ignores diversification — deliberately so; we discuss why in the debrief.)
# TODO 3: print only the tickers whose VaR exceeds $500,000 (an if inside the loop).

## ⚡ Lab B fast lane (optional)
1. Sort the book by position VaR, largest first, *without pandas* (hint: `sorted(book, key=..., reverse=True)`).
2. Rewrite the VaR loop as a dictionary comprehension: `var_by_ticker = {t: position_var(v, vols[t]) for t, v in book.items()}`. The goal is a dict of **numbers** you can keep computing with — not printed text. Prove yours works by computing `sum(var_by_ticker.values())` and looking up one ticker, e.g. `var_by_ticker["HLT"]`. (Printing pretty output is fine, but it's a separate second step — if your comprehension produces formatted strings, you've lost the numbers.)
3. Extend the `position_pnl` function (two cells up) with a third parameter `stop_loss=-0.02` (a return threshold with a default value), and make the function also return `True`/`False` for whether the loss breached it.

## Lab B check-out (2 min)
One sentence: *name a calculation you repeat across many rows or tabs at work/school — what would the function be called?*

_your answer here_

In [66]:
book = {
    "HLT" : 25_000_000,
    "MAR" : 18_000_000,
    "H" : 9_500_000
}
book["CHH"] = 4_200_000
print(book["MAR"])
total = sum(book.values())
print(total)

18000000
56700000


---
# Lab C · pandas: the spreadsheet that fits in a variable (1:30–2:10)

You just watched the live pivot & VLOOKUP demo (`01b_excel_to_pandas_demo.ipynb` — it stays in the repo; re-read it any time). Now the same moves on the real FCC data you'll use in Case 1.

A **DataFrame** is a table. Every Excel operation you know has a one-line equivalent:

| Excel | pandas |
|---|---|
| Open file | `pd.read_csv(...)` |
| Filter rows | `df[df["col"] > 100]` |
| New calculated column | `df["new"] = df["a"] * df["b"]` |
| SUM / AVERAGE / COUNT | `df["col"].sum()`, `.mean()`, `.count()` |
| Pivot table | `df.groupby("key")["col"].agg(["sum","mean"])` |
| VLOOKUP / XLOOKUP | `df.merge(other, on="key", how="left")` |
| Remove duplicates | `df.drop_duplicates()` |
| Sort | `df.sort_values("col", ascending=False)` |
| Save as | `df.to_excel("out.xlsx")` |

In [12]:
import pandas as pd
avail = pd.read_csv(f"{DATA}/case1/bdc_il_fixed_availability_sample.csv.gz", dtype={"frn": str, "block_geoid": str})
print(avail.shape)          # (rows, columns)
avail.head()

(236549, 12)


,frn,provider_id,brand_name,location_id,technology,max_advertised_download_speed,max_advertised_upload_speed,low_latency,business_residential_code,state_usps,block_geoid,h3_res8_id
0,0031200006,130006,OrbitNet,1000025989,60,100,12,0,R,IL,170310243001062,8819343707ffff
1,0031200006,130006,OrbitNet,1000015517,60,100,12,0,R,IL,170890343001118,8818964588ffff
2,0031200002,130002,Prairie Fiber,1000046910,50,2000,2000,1,R,IL,172010887001062,8823518080ffff
3,0031200002,130002,Prairie Fiber,1000004199,50,500,500,1,R,IL,170970104001033,8821430841ffff
4,0031200006,130006,OrbitNet,1000039916,60,100,12,0,X,IL,170430368001182,8816772892ffff


In [13]:
avail.info()                # column names, types, missing values — always look at this first

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 236549 entries, 0 to 236548
Data columns (total 12 columns):
 #   Column                         Non-Null Count   Dtype 
---  ------                         --------------   ----- 
 0   frn                            236549 non-null  object
 1   provider_id                    236549 non-null  int64 
 2   brand_name                     236549 non-null  object
 3   location_id                    236549 non-null  int64 
 4   technology                     236549 non-null  int64 
 5   max_advertised_download_speed  236549 non-null  int64 
 6   max_advertised_upload_speed    236549 non-null  int64 
 7   low_latency                    236549 non-null  int64 
 8   business_residential_code      236549 non-null  object
 9   state_usps                     236549 non-null  object
 10  block_geoid                    236549 non-null  object
 11  h3_res8_id                     236549 non-null  object
dtypes: int64(6), object(6)
memory usage: 21.7+ M

In [14]:
# Filter: which rows are fiber (technology code 50)?
fiber = avail[avail["technology"] == 50]
print(f"{len(fiber):,} fiber offers out of {len(avail):,} rows ({len(fiber)/len(avail):.1%})")

28,866 fiber offers out of 236,549 rows (12.2%)


In [15]:
# Pivot: how many offers does each provider have, by technology?
pivot = avail.pivot_table(index="brand_name", columns="technology", values="location_id", aggfunc="count", fill_value=0)
pivot

technology,10,40,50,60,61,70
brand_name,,,,,,
HERITAGETEL,388,0,62,0,0,0
HeritageTel,39324,0,5308,0,0,0
HeritageTel,393,0,57,0,0,0
LAKESHORE BROADBAND,0,55,30,0,0,0
Lakeshore Broadband,0,6996,3653,0,0,0
Lakeshore Broadband,0,70,40,0,0,0
MIDSTATE CABLE,0,349,0,0,0,0
MidState Cable,0,34591,0,0,0,0
MidState Cable,0,326,0,0,0,0


In [16]:
# Hmm — why do some providers appear twice in the pivot above?
# The table display hides the culprits — print the raw values WITH their quotes to see them:
sorted(avail["brand_name"].unique())

['HERITAGETEL',
 'HeritageTel',
 'HeritageTel  ',
 'LAKESHORE BROADBAND',
 'Lakeshore Broadband',
 'Lakeshore Broadband  ',
 'MIDSTATE CABLE',
 'MidState Cable',
 'MidState Cable  ',
 'ORBITNET',
 'OrbitNet',
 'OrbitNet  ',
 'PRAIRIE FIBER',
 'Prairie Fiber',
 'Prairie Fiber  ',
 'SKYREACH WIRELESS',
 'STARMESH',
 'SkyReach Wireless',
 'SkyReach Wireless  ',
 'StarMesh',
 'StarMesh  ']

In [17]:
# TODO 1: make the brand names consistent. Hint: avail["brand_name"] = avail["brand_name"].str.strip().str.title()
#         then re-run the unique() above — the duplicates should collapse from 21 variants to 7 brands.
#         But look closely at the survivors: .title() lowercases mid-word capitals, so 'MidState Cable'
#         became 'Midstate Cable' and 'HeritageTel' became 'Heritagetel' — merged, but misbranded.
#         Fix those with .replace({"Midstate Cable": "MidState Cable", ...}) — real-world cleaning
#         is usually a rule (strip/title) plus a small manual exception list. That is normal.

# TODO 2: count how many offers meet the 100/20 Mbps threshold:
#         both max_advertised_download_speed >= 100 AND max_advertised_upload_speed >= 20.
#         Hint: combine two conditions with & and wrap each in parentheses.

In [18]:
# VLOOKUP → merge: attach the technology name to every row
tech = pd.read_csv(f"{DATA}/case1/bdc_technology_codes.csv")
avail = avail.merge(tech, on="technology", how="left")
avail[["brand_name", "technology", "technology_name"]].drop_duplicates().head(10)

,brand_name,technology,technology_name
0,OrbitNet,60,Geostationary Satellite
2,Prairie Fiber,50,Fiber to the Premises
5,MidState Cable,40,Cable (Coaxial / HFC)
6,SkyReach Wireless,70,Unlicensed Fixed Wireless
7,HeritageTel,10,Copper Wire (DSL)
9,StarMesh,61,Non-geostationary Satellite
15,HeritageTel,50,Fiber to the Premises
24,Lakeshore Broadband,40,Cable (Coaxial / HFC)
33,Lakeshore Broadband,50,Fiber to the Premises
91,StarMesh,61,Non-geostationary Satellite


In [19]:
# Save your work as an Excel file — Python is not a replacement for Excel as a *delivery* format.
avail.head(1000).to_excel("bootcamp_sample.xlsx", index=False)
print("Saved bootcamp_sample.xlsx — open the file browser on the left to download it.")

Saved bootcamp_sample.xlsx — open the file browser on the left to download it.


## ⚡ Lab C fast lane (optional)
1. Plot the number of offers per technology as a bar chart: `avail["technology_name"].value_counts().plot(kind="bar")`.
2. What share of rows are exact duplicates? (`avail.duplicated().sum()`) Why might a provider file the same location twice?
3. Write a function `served(df, down, up)` that returns the subset meeting any threshold, and call it with (25, 3) and (100, 20).
4. **One-expression rebuild.** Everything you did above happened as separate steps that each
   overwrote `avail`: load the CSV → clean the brand names (strip, title-case, fix the two
   CamelCase brands) → merge on the technology names → filter to offers meeting 100/20 Mbps →
   pivot offers per brand × technology. Now produce that same final pivot as **one chained
   expression** starting from a fresh `pd.read_csv(...)`, with no intermediate variables and
   no re-assigning `avail`. Skeleton to complete:

```python
result = (
    pd.read_csv(f"{DATA}/case1/bdc_il_fixed_availability_sample.csv.gz", dtype={"frn": str, "block_geoid": str})
      .drop_duplicates()
      .assign(brand_name=lambda d: ...)   # the strip/title/replace cleaning, as one expression
      .merge(tech, on="technology", how="left")
      .query("...")                        # the 100/20 threshold
      .pivot_table(...)                    # offers per brand_name × technology_name
)
```

   Why bother? A chain reads top-to-bottom like a recipe, nothing in between can be stale or
   half-updated (a real hazard above — re-run the merge cell twice and you silently get
   duplicate `technology_name_x`/`_y` columns),
   and it's the shape most production pandas code takes.

## Lab C check-out (2 min)
Write one sentence in the cell below: *what is one thing you did here that would have been painful in Excel?*

_your answer here_